# Euphoria Machine Learning Analysis

This notebook presents the cleaned portfolio workflow for the original group academic project.

**Objectives**
1. Predict the continuous `happiness_index` target with regression.
2. Explore island segmentation with K-Means clustering.

The raw dataset contains unusual CSV formatting, so loading is handled by the robust parser in `src/data_cleaning.py`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_cleaning import load_raw_dataset, engineer_features, prepare_regression_frame
from src.modeling import run_regression
from src.clustering import run_clustering

RAW_PATH = ROOT / "data" / "raw" / "euphoria_dataset.csv"
OUTPUT_DIR = ROOT / "outputs"


## 1. Load and inspect the dataset

In [2]:
df = load_raw_dataset(RAW_PATH)
df.shape


(99492, 19)

In [3]:
engineered = engineer_features(df)
summary = pd.DataFrame({
    "dtype": engineered.dtypes.astype(str),
    "missing_pct": engineered.isna().mean().mul(100).round(2),
    "n_unique": engineered.nunique(dropna=True)
})
summary.sort_values("missing_pct", ascending=False).head(12)


,dtype,missing_pct,n_unique
fauna_friendly,object,64.66,4
features,object,24.58,9404
nearest_city,object,10.29,2877
region,object,10.24,51
entry_fee,object,10.18,2
shelters,float64,10.16,9
y_coordinate,float64,10.15,7034
happiness_index,float64,10.09,3605
creation_time,float64,10.08,69707
creation_year,float64,10.08,2


## 2. Problem formulation

The original notebook mixed classification and regression language. In the supplied data, `happiness_index` is continuous, so the corrected supervised-learning task is **regression**.


In [4]:
engineered["happiness_index"].describe()


count    89455.000000
mean      1527.617361
std        896.880412
min        100.000000
25%       1015.000000
50%       1350.000000
75%       1795.000000
max      52500.000000
Name: happiness_index, dtype: float64

## 3. Regression models

In [5]:
regression = run_regression(df, OUTPUT_DIR)
regression.metrics


,model,MAE,RMSE,R2
0,Extra Trees,243.221210,531.474299,0.657608
1,XGBoost,265.871434,522.461287,0.669122
2,Median baseline,529.681069,926.043414,-0.039495


The comparison includes a median baseline plus two nonlinear models. Preprocessing is fitted only on the training partition to avoid leakage.


In [6]:
regression.feature_importance.head(15)


,feature,importance
8,y_coordinate,0.256040
14,region,0.174421
3,island_size,0.117817
6,x_coordinate,0.114877
1,water_sources,0.081671
16,nearest_city,0.056776
2,shelters,0.036058
10,creation_month,0.035437
11,amenity_count,0.028544
4,loyalty_score,0.019108


## 4. Unsupervised segmentation

In [7]:
clustering = run_clustering(df, OUTPUT_DIR)
clustering.scores


,k,silhouette,davies_bouldin
0,2,0.159690,2.217196
1,3,0.116976,2.369224
2,4,0.102975,2.668858
3,5,0.092155,2.565859
4,6,0.091015,2.420606


In [8]:
print("Selected k:", clustering.best_k)
clustering.profiles


Selected k: 2


,count,referral_friends,water_sources,shelters,island_size,happiness_index,loyalty_score,total_refunds_requested,avg_time_in_euphoria,x_coordinate,y_coordinate,amenity_count
cluster,,,,,,,,,,,,
0,40389,1.5,2.00,2.33,1216.41,1777.79,5.51,3.0,60.01,36.56,-91.29,3.35
1,59103,1.5,1.02,1.32,764.93,1323.10,5.52,3.0,59.80,37.19,-91.72,3.23


## 5. Interpretation

- Both tree-based regressors outperform the median baseline on the held-out test set.
- XGBoost has the strongest RMSE/R² in the reproducible run, while Extra Trees has the lowest MAE.
- K-Means cluster separation is modest (silhouette around 0.15), so the segmentation is exploratory rather than evidence of sharply separated natural groups.
- The portfolio edition emphasizes reproducibility, leakage prevention, transparent baselines, and cautious interpretation.
